# Préparation des données DPO — MediQAl (mcqm)

Construction d'un jeu de paires préférentielles **en français** (`prompt` / `chosen` / `rejected`) pour l'alignement DPO, à partir des QCM à réponses multiples de MediQAl (configuration `mcqm`, licence CC-BY-4.0).

- **prompt** : cas clinique, question, puis les propositions A à E, et la consigne « Plusieurs propositions sont exactes (au moins deux). ».
- **chosen** : toutes les bonnes propositions (lettre et texte), d'après le corrigé officiel.
- **rejected** : la même liste avec **une erreur**, construite à partir des distracteurs (mauvaises options) rédigés par des enseignants en médecine. L'erreur est donc plausible, et le corrigé garantit qu'elle est fausse.
- **Qualité plutôt que quantité** : une seule paire par question, uniquement les questions rattachées à un cas clinique.
- **Répartition par cas clinique** (`projet_14.mediqal`) : ce notebook ne garde que les cas réservés au DPO. Le SFT (`mcqu` et `mcqm`) utilise les autres : aucun cas n'est vu à la fois en SFT et en DPO, ni à la fois à l'entraînement et à l'évaluation.
- **Sortie** : `data/anonymized/dpo_mediqal_mcqm_{train,validation,test}.anonymized.jsonl`, au format conversationnel attendu par `trl.DPOTrainer`.

# Import

In [1]:
import json
import re

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

from projet_14.mediqal import (
    LETTERS, OPTION_COLUMNS, OPTIONS_HEADER, SYSTEM_PROMPT, assign_cases, clean_text, correct_letters, format_answer,
    user_content,
)
from projet_14.rgpd import run

SEED = 42

# Taille maximale des jeux après filtrage (le test est gardé entier)
N_TRAIN = 2000
N_VALIDATION = 300

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqm")
# Les trois splits officiels réunis : le découpage est refait par cas clinique (plus bas)
df = pd.concat([ds[name].to_pandas().assign(official_split=name) for name in ds], ignore_index=True)
df_train = df[df["official_split"] == "train"]

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df_train.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type,official_split
0,11399,Plusieurs jours après un contage avec un enfan...,Quelle(s) proposition(s) peut (peuvent) s'appl...,Transmission manuportée,La phase de contagiosité dure 8 jours,Maladie strictement humaine,Confère une immunité,Transmission indirecte possible,"C,D",QCM,Infectious Diseases,Understanding,train
1,17928,NaN,Quel(s) germe(s) est (sont) responsable(s) de ...,Gardnerella Vaginalis,Trichomonas,Tréponèmes,Candida albicans,Strepto B,"A,B",QCM,Gynecology and Obstetrics,Understanding,train
2,17520,"Une jeune femme de 26 ans, d'origine française...","En cours d'hospitalisation, laquelle ou lesque...",Réduction de l'apport en protéines animales,Association d'une sulfonylurée à l'insuline,Conservation du même type d'insuline mais augm...,Education à l'autosurveillance glycémique,"Adaptation d'un nouveau schéma thérapeutique, ...","A,D,E",QCM,Endocrinology and Metabolism,Reasoning,train
3,10875,NaN,Quelles sont parmi les bactéries suivantes res...,Staphylococcus aureus,Salmonella enteritidis,Shigella sonnei,Vibrio cholerae,Campylobacter fetus,"B,C,E",QCM,Pediatrics,Understanding,train
4,14691,NaN,"Parmi les 5 propositions suivantes, quelle est...",Il est plus souvent d'origine maternelle,La majorité des formes sont inapparentes,Une primo-infection génitale maternelle durant...,L'accouchement par césarienne est indiqué lors...,Aucune chimiothérapie antivirale n'est active,"A,C,D",QCM,Pediatrics,Understanding,train


In [4]:
# Nombres de lignes dans les différentes parties du dataset
pd.crosstab(df["official_split"], df["clinical_case"].notna().map({True: "avec cas", False: "sans cas"}))

clinical_case,avec cas,sans cas
official_split,,
test,1342,2042
train,2185,3582
validation,547,919


In [5]:
# Nombre de bonnes réponses par question
n_correct = df_train["correct_answers"].str.split(",").map(len)
n_correct.value_counts().sort_index().rename_axis("bonnes réponses").to_frame("questions")

,questions
bonnes réponses,
2,2054
3,2234
4,1154
5,325


In [6]:
# Valeurs distinctes de "question_type" et "medical_subject"
print(df_train["question_type"].value_counts(), end="\n\n")
print(df_train["medical_subject"].value_counts())

question_type
Understanding    4320
Reasoning        1447
Name: count, dtype: int64

medical_subject
Cardiology                          395
Hepato-Gastroenterology             395
Pulmonology                         341
Psychiatry                          329
Nephro-Urology                      328
Endocrinology and Metabolism        300
Infectious Diseases                 291
Gynecology and Obstetrics           290
Rheumatology                        289
Hematology                          281
Neurology                           278
Dermatology                         270
Pediatrics                          249
Otorhinolaryngology (ENT)           220
Ophthalmology                       196
Forensic Medicine and Toxicology    172
Immunology                          158
Pharmacology                        147
Biochemistry                        142
Pathological Anatomy                141
Orthopedics                         127
Microbiology                        100
Epidemiology       

In [7]:
# Longueur des options (en caractères)
options = pd.concat([df_train[column] for column in OPTION_COLUMNS])
options.str.len().describe()

count    28835.000000
mean        33.046818
std         21.357796
min          1.000000
25%         18.000000
50%         28.000000
75%         43.000000
max        166.000000
dtype: float64

# Nettoyage

In [8]:
TEXT_COLUMNS = ["clinical_case", "question"] + OPTION_COLUMNS
df = df.assign(**{col: df[col].map(clean_text, na_action="ignore") for col in TEXT_COLUMNS})

## Répartition par cas clinique

MediQAl n'a pas d'identifiant de cas, et ses splits officiels partagent des cas (un même cas sert à plusieurs questions). `assign_cases` calcule un identifiant (hash du texte du cas) et en tire un **usage** (`sft` ou `dpo`) et un **split** (70 % train, 10 % validation, 20 % test), identiques dans tous les notebooks MediQAl.

Les options « méta » (« Aucune des propositions précédentes ») sont gardées : les propositions sont maintenant dans le prompt, elles ont donc un sens.

In [9]:
# Uniquement les questions rattachées à un cas clinique (contexte patient, proche du triage)
n_before = len(df)
df = df.dropna(subset=["clinical_case", "question"]).drop_duplicates(subset=["clinical_case", "question"])
print(f"{n_before} rows, {len(df)} avec un cas clinique (après dédoublonnage)")

# Répartition par cas clinique : on garde les cas réservés au DPO, dans leur split
df = assign_cases(df)
print(pd.crosstab(df["usage"], df["split"], margins=True), end="\n\n")
splits = {name: df[(df["usage"] == "dpo") & (df["split"] == name)] for name in ["train", "validation", "test"]}
for name, d in splits.items():
    print(f"{name}: {len(d)} questions, {d['case_id'].nunique()} cas")

10617 rows, 4066 avec un cas clinique (après dédoublonnage)
split  test  train  validation   All
usage                               
dpo     481   1357         220  2058
sft     431   1398         179  2008
All     912   2755         399  4066

train: 1357 questions, 521 cas
validation: 220 questions, 77 cas
test: 481 questions, 169 cas


# Construction des paires DPO (chosen / rejected)

La réponse **chosen** est la liste des bonnes réponses. La réponse **rejected** contient une seule erreur, d'un des trois types suivants (tiré au hasard parmi ceux possibles) :

| Type | Construction | Erreur clinique simulée |
|---|---|---|
| `omission` | une bonne réponse retirée | oublier un élément (un signe de gravité, un examen) |
| `ajout` | un distracteur ajouté | retenir un élément faux |
| `substitution` | une bonne réponse remplacée par un distracteur | se tromper d'élément, à longueur égale |

Les trois types se compensent : `omission` raccourcit la réponse, `ajout` l'allonge, `substitution` garde le même nombre d'éléments. Le modèle ne peut donc pas apprendre « plus long = meilleur » (ni l'inverse). Les éléments sont toujours affichés dans l'ordre des options (A à E), avec leur lettre (« - A. … ») : la position ne trahit pas l'erreur.

In [10]:
rng = np.random.default_rng(SEED)


def render(letters, row) -> str:
    # Liste à puces « - A. texte », dans l'ordre A à E (même format que les réponses multiples du SFT)
    return format_answer(row, letters, as_list=True)


def make_pair(row) -> pd.Series:
    correct = set(correct_letters(row))
    wrong = sorted(set(LETTERS) - correct)
    removed = str(rng.choice(sorted(correct)))
    added = str(rng.choice(wrong)) if wrong else None

    candidates = {
        "omission": correct - {removed},
        "ajout": correct | {added},
        "substitution": (correct - {removed}) | {added},
    }
    if not wrong:  # toutes les options sont justes : seule l'omission est possible
        candidates = {"omission": candidates["omission"]}
    rejected_type = str(rng.choice(list(candidates)))
    return pd.Series(
        {
            "chosen": render(correct, row),
            "rejected": render(candidates[rejected_type], row),
            "rejected_type": rejected_type,
        }
    )


for name, df in splits.items():
    df = pd.concat([df, df.apply(make_pair, axis=1)], axis=1)
    splits[name] = df[df["chosen"] != df["rejected"]]

pd.concat({name: df["rejected_type"].value_counts() for name, df in splits.items()}, axis=1)

,train,validation,test
rejected_type,,,
omission,495,80,176
substitution,434,63,155
ajout,428,77,150


In [11]:
# Un exemple de chaque type
for rejected_type in ["omission", "ajout", "substitution"]:
    example = splits["train"][splits["train"]["rejected_type"] == rejected_type].iloc[0]
    print(f"===== {rejected_type}\nPROMPT : …{user_content(example, multiple_answers=True)[-500:]}\n")
    print(f"CHOSEN :\n{example['chosen']}\n\nREJECTED :\n{example['rejected']}\n")

===== omission
PROMPT : …après quinze jours de traitement digitalo-diurétique et confirme le diagnostic clinique. Ce bilan révèle :

Propositions :
A. Index cardiaque à 4 l/mn/m2
B. Aspect de ventricularisation de la courbe auriculaire droite inspiration
C. Gradient de 50 mmHg au retrait ventriculo-aortique
D. Pression capillaire pulmonaire moyenne à 25 mmHg avec télédiastolique ventriculaire gauche à 10 mmHg
E. Pression télédiastolique ventriculaire droite à 15 mmHg

Plusieurs propositions sont exactes (au moins deux).

CHOSEN :
- B. Aspect de ventricularisation de la courbe auriculaire droite inspiration
- D. Pression capillaire pulmonaire moyenne à 25 mmHg avec télédiastolique ventriculaire gauche à 10 mmHg
- E. Pression télédiastolique ventriculaire droite à 15 mmHg

REJECTED :
- D. Pression capillaire pulmonaire moyenne à 25 mmHg avec télédiastolique ventriculaire gauche à 10 mmHg
- E. Pression télédiastolique ventriculaire droite à 15 mmHg

===== ajout
PROMPT : …ux, mais pas de d

In [12]:
# Biais de longueur : part des paires où la chosen est la plus longue
for name, df in splits.items():
    longer = (df["chosen"].str.len() > df["rejected"].str.len()).mean()
    print(f"{name}: chosen plus longue dans {longer:.0%} des paires")

train: chosen plus longue dans 53% des paires
validation: chosen plus longue dans 49% des paires
test: chosen plus longue dans 51% des paires


# Échantillonnage

Le découpage train / validation / test vient de la répartition par cas clinique. Qualité plutôt que quantité : on plafonne le train à `N_TRAIN` paires et la validation à `N_VALIDATION`. Le test est gardé entier.

In [13]:
splits["train"] = splits["train"].sample(n=min(N_TRAIN, len(splits["train"])), random_state=SEED)
splits["validation"] = splits["validation"].sample(n=min(N_VALIDATION, len(splits["validation"])), random_state=SEED)

for name, df in splits.items():
    print(f"{name}: {len(df)} rows")

train: 1357 rows
validation: 220 rows
test: 481 rows


# Build DPO JSONL file from dataset

In [14]:
# Même system prompt que pour le SFT : le DPO part du modèle SFT et doit voir le même gabarit
def to_prompt(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content(row, multiple_answers=True)},  # cas clinique, question, propositions, consigne
    ]


def to_assistant(text: str) -> list[dict]:
    return [{"role": "assistant", "content": text}]


METADATA = ["id", "source", "case_id", "medical_subject", "question_type", "rejected_type", "pair_check", "golden_answer", "lang"]

# Un fichier JSONL par split : ../data/dpo_mediqal_mcqm_{train,validation,test}.jsonl
dpo_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(
        source="MediQAl-mcqm",
        pair_check="golden_answer",  # rejected construit à partir du corrigé officiel
        golden_answer=df["correct_answers"],
        lang="fr",
        prompt=df.apply(to_prompt, axis=1),
        chosen=df["chosen"].map(to_assistant),
        rejected=df["rejected"].map(to_assistant),
    )
    dpo_ds[name] = Dataset.from_pandas(df[METADATA + ["prompt", "chosen", "rejected"]], preserve_index=False)
    dpo_ds[name].to_json(f"../data/dpo_mediqal_mcqm_{name}.jsonl", force_ascii=False)

dpo_ds["train"][0]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 36.18ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 99.27ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 49.82ba/s]

{'id': '21042',
 'source': 'MediQAl-mcqm',
 'case_id': '34324c443d30',
 'medical_subject': 'Orthopedics',
 'question_type': 'Reasoning',
 'rejected_type': 'ajout',
 'pair_check': 'golden_answer',
 'golden_answer': 'C,D',
 'lang': 'fr',
 'prompt': [{'role': 'system',
   'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
  {'role': 'user',
   'content': "Monsieur V..., âgé de 62 ans, barman à la retraite, consulte pour l'apparition progressive depuis 2 mois de radiculalgies diffuses, insomniantes. Il se plaint d'une grande asthénie, a maigri de 5 kilos et doit s'arrêter au bout d'un périmètre de marche de 400 m du fait d'une sensation de faiblesse des membres inférieurs. Son médecin lui prescrit du Primpéran® en raison d'importantes nausées.Dans ses antécédents on relève une intoxication alcoolo-tabagique (40 paquets année; 1 litre de vin par jour), il a subi il y a un an une résection endoscopique de prostate pour un adénome. Le patient se lève 3 fois

In [15]:
print(dpo_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 1357
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 220
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 481
    })
})


# Anonymisation des données

On n'anonymise que le **cas clinique et la question** (champ `prompt`, jusqu'à l'en-tête des propositions : `keep_after`), qui décrivent le patient. Les propositions et les réponses `chosen` / `rejected` sont des options de QCM rédigées par les enseignants : des énoncés médicaux courts, sans donnée patient. Presidio y prend presque chaque terme en début de ligne pour un nom (« Angiographie », « Pyélonéphrite », « Érythrose palmaire »…) : anonymiser les options effacerait les éléments que la paire compare, et pourrait même rendre `chosen` et `rejected` identiques.

Contrôle de compensation : la cellule suivante liste les options qui contiennent une civilité suivie d'un nom, à relire à la main.

In [16]:
# Options contenant une civilité suivie d'un nom (« Monsieur X. », « Mme Dupont ») : à relire
TITLE_IN_OPTION = re.compile(r"\b(?:Monsieur|Madame|Mademoiselle|Mme|Mlle|M\.|Mr|Dr)\s+[A-ZÀ-Ý]")
flagged = [
    (name, row["id"], text)
    for name, df in splits.items()
    for _, row in df.iterrows()
    for text in (row["chosen"], row["rejected"])
    if TITLE_IN_OPTION.search(text)
]
print(f"{len(flagged)} réponses à relire")
pd.DataFrame(flagged, columns=["split", "id", "réponse"]).drop_duplicates().head(20)

7 réponses à relire


,split,id,réponse
0,train,5098,- B. Sérothérapie spécifique pour Mlle X...\n-...
1,train,17559,- C. Il existe une hépathopathie. Monsieur B. ...
2,train,17559,- C. Il existe une hépathopathie. Monsieur B. ...
3,test,6818,- B. Indemnités journalières au cours du congé...
4,test,6818,- B. Indemnités journalières au cours du congé...
5,test,12740,- A. La mère de Monsieur X. est asthmatique\n-...
6,test,12740,- A. La mère de Monsieur X. est asthmatique\n-...


In [17]:
# Anonymisation de chaque split : ../data/anonymized/dpo_mediqal_mcqm_{split}.anonymized.jsonl
reports = {
    name: run(f"../data/dpo_mediqal_mcqm_{name}.jsonl", lang="fr", output_dir="../data/anonymized", fields=["prompt"],
        keep_after=OPTIONS_HEADER)
    for name in dpo_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,1357,424,752
validation,220,82,147
test,481,168,336


In [18]:
# DataFrame des champs "content" des jsonl anonymisés : une ligne par message (hors system)
rows = []
for name in dpo_ds:
    with open(f"../data/anonymized/dpo_mediqal_mcqm_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for field in ["prompt", "chosen", "rejected"]:
                for message in record[field]:
                    if message["role"] != "system":
                        rows.append({"split": name, "id": record["id"], "field": field, "content": message["content"]})

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
contents_df.groupby("field")["n_person"].agg(["sum", lambda s: (s > 0).sum()]).set_axis(["balises", "messages"], axis=1)

6174 messages, dont 674 avec au moins une balise <PERSON>


,balises,messages
field,,
chosen,0,0
prompt,992,674
rejected,0,0


In [19]:
def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["split", "id", "field"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

992 occurrences de <PERSON>


,split,id,field,context
1395,train,9620,prompt,"arctus du myocarde à 47 ans. Un bilan biologique est fait à <PERSON>, après 1 à 2 heures de jeûne, qui montre : cholestérolémie"
2151,train,3140,prompt,"Monsieur <PERSON>, 69 ans, consulte pour une anémie et des troubles de la mar"
3189,train,8364,prompt,"étion, un rein droit de 12,5 cm et un rein gauche de 12 cm. <PERSON> n'y a ni protéinurie, ni hématurie, ni HTA, ni oedème et ni"
2046,train,13693,prompt,"Monsieur <PERSON>.. 43 ans, ingénieur est amené en urgence dans le service de"
615,train,3265,prompt,"on de leur volume est normal. L'examen clinique est normal. <PERSON> : - globules rouges 3 500 000/mm3- hémoglobine = 7,5 g/dl-"
3468,train,11427,prompt,de est porté Le syndrome douloureux abdominal que présente <PERSON> s'est accompagné. dans les heures qui ont suivi l'hospitali
1851,train,16111,prompt,"cuité visuelle gauche. Ses antécédents sont marqués par une <PERSON> reconnue il y a 2 ans, traitée par inhibiteur calcique et b"
735,train,21106,prompt,nt à 133 % de son poids théorique idéal selon la formule de <PERSON>. Il s'est fait faire un bilan biologique dont les résultats
405,train,6728,prompt,"Interné au pavillon d'urgence, un homme, monsieur <PERSON>.. blessé par balles, se présente à vous. Il déclare avoir é"
3249,train,11686,prompt,"parties molles, aspect d'excentration de la tête fémorale <PERSON>) séquelle(s) peut-on craindre ? Propositions : A. Aucune B"


# Contrôle final

In [20]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement DPO
DATA_FILES = {name: f"../data/anonymized/dpo_mediqal_mcqm_{name}.anonymized.jsonl" for name in dpo_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

for name, split in final_ds.items():
    same = sum(c[0]["content"] == r[0]["content"] for c, r in zip(split["chosen"], split["rejected"]))
    assert same == 0, f"{name}: {same} paires avec chosen == rejected"

# Aucun cas clinique partagé entre train, validation et test
cases = {name: set(split["case_id"]) for name, split in final_ds.items()}
assert not cases["train"] & (cases["validation"] | cases["test"]), "cas partagé entre train et évaluation"
assert not cases["validation"] & cases["test"], "cas partagé entre validation et test"
print("OK : aucune paire avec chosen == rejected, aucun cas partagé entre les splits")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 1357 examples [00:00, 84298.57 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 220 examples [00:00, 36563.26 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 481 examples [00:00, 50749.89 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 1357
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 220
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'rejected_type', 'pair_check', 'golden_answer', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 481
    })
})
OK : aucune paire avec chosen == rejected, aucun cas partagé entre les splits


# Limites

- **Réponses courtes, sans justification** : les paires apprennent *quoi* répondre, pas à *expliquer* la réponse. Les explications viennent de la partie anglaise (UltraMedical).
- **Erreurs simulées** : les `rejected` sont construits, pas produits par le modèle. Étape suivante possible : faire répondre le modèle SFT à ces questions et prendre ses réponses fausses (vérifiables par le corrigé) comme `rejected`.